## Funding analysis (2020-2025)

Computes the standard breakdowns needed for reporting, exported as CSVs sized to paste directly
into the existing Google Sheets template (`raw_data/Funding analysis 2026.xlsx`, 'Europe wide'
tab) - keeps chart formatting consistent with prior reports while the underlying numbers are
computed in code.

Methodology confirmed against that template rather than assumed:
- Funding figure throughout = `'Gov & NP contribution (EUR)'`.
- Excludes any grant whose `'Award purpose'` is *exactly* `'Equipment and infrastructure'` (not
  combined with anything else).
- "Country" = `'Funder Country'` (whose money it is), not `'PI organisation country'`.
- Multi-value cells (countries, pillars, end products) split the grant's funding evenly across
  the **distinct** values listed.
- Growth rate = simple year-over-year % change.
- Population/GDP from the World Bank, R&D spend from OECD - fetched live (see
  `Funding_analysis_helpers.py`).

In [39]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from Funding_analysis_helpers import (
    split_multi_value, fetch_worldbank_indicator, fetch_oecd_gerd, fetch_usd_to_eur_rate,
    lookup_country_value, COUNTRY_NAME_TO_OECD, yoy_growth, YEAR_COLS,
)

FINAL_FILE = Path('data_output/260727_0922_ap_pillar_backfill_funding_curated_FINAL.xlsx')
OUTPUT_DIR = Path('data_output/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUNDING_COL = 'Gov & NP contribution (EUR)'
YEAR_COL_STRS = [str(y) for y in YEAR_COLS]
ALL_TIME_YEARS = [str(y) for y in range(2020, 2026)]  # 2020-2025, matching the template's "All time" window

# Tied to FINAL_FILE's own run-timestamp prefix, not "now" - change FINAL_FILE above to analyze a
# different run and this follows automatically. Reruns of this notebook overwrite the same CSVs
# instead of piling up a fresh timestamped copy every time.
RUN_TIMESTAMP = re.match(r'(\d{6}_\d{4})', FINAL_FILE.name).group(1)


def save(df_or_series, name, **to_csv_kwargs):
    path = OUTPUT_DIR / f'{RUN_TIMESTAMP}_{name}.csv'
    df_or_series.to_csv(path, **to_csv_kwargs)
    print(f"Saved {name} -> {path}")

### Load and filter

Loads the curated dataset and applies the pipeline-wide filtering rules below - every analysis in
this notebook operates on the filtered `df`, not the raw `FINAL_FILE` export directly. Two filters:
1. Drop grants whose `'Award purpose'` is *only* `'Equipment and infrastructure'`.
2. Drop grants whose `'Funding decision'` is `'Committed'` (pledged, not yet actually awarded or not the offical grant(s)).

In [40]:
df = pd.read_excel(FINAL_FILE)
print(f"Loaded {len(df)} rows from '{FINAL_FILE.name}'.")

# Filter 1: exclude grants whose Award purpose is ONLY 'Equipment and infrastructure' - a grant
# combining it with something else (e.g. 'Research and development; Equipment and infrastructure')
# stays in. Confirmed against the colleagues' own 'Europe wide' template, which applies the same rule.
award_purpose_mask = df['Award purpose'].astype(str).str.strip() == 'Equipment and infrastructure'
df = df[~award_purpose_mask].reset_index(drop=True)
print(f"Excluded {award_purpose_mask.sum()} grants whose Award purpose is ONLY 'Equipment and infrastructure'.")

# Filter 2: exclude grants whose Funding decision is 'Committed' - money that's been pledged but
# not yet actually awarded shouldn't count in a funding-received analysis. This is a general rule,
# not a one-off exclusion - it also removes the €162.5M Danish government commitment the
# colleagues' template excludes from every total (confirmed its own Funding decision is
# 'Committed' too), plus every other Committed-but-not-yet-awarded grant in the dataset.
committed_mask = df['Funding decision'].astype(str).str.strip() == 'Committed'
df = df[~committed_mask].reset_index(drop=True)
print(f"Excluded {committed_mask.sum()} grants whose Funding decision is 'Committed'.")

print(f"{len(df)} grants remain for analysis.")

Loaded 2481 rows from '260727_0922_ap_pillar_backfill_funding_curated_FINAL.xlsx'.
Excluded 27 grants whose Award purpose is ONLY 'Equipment and infrastructure'.
Excluded 17 grants whose Funding decision is 'Committed'.
2437 grants remain for analysis.


### 1. Total funding by start year, and by funding year

In [41]:
by_start_year = df[df['Year project started'].between(2020, 2025)].groupby('Year project started')[FUNDING_COL].sum()
by_start_year.index = by_start_year.index.astype(int)
by_start_year.name = 'Total funding (EUR)'
save(by_start_year, '1a_funding_by_start_year')
print(by_start_year)

by_funding_year = df[YEAR_COL_STRS].sum()
by_funding_year.name = 'Total funding (EUR)'
save(by_funding_year, '1b_funding_by_funding_year')
print()
print(by_funding_year)

Saved 1a_funding_by_start_year -> data_output\analysis\260727_0922_1a_funding_by_start_year.csv
Year project started
2020    1.374401e+08
2021    2.320233e+08
2022    2.817690e+08
2023    4.286631e+08
2024    4.892599e+08
2025    3.178548e+08
Name: Total funding (EUR), dtype: float64
Saved 1b_funding_by_funding_year -> data_output\analysis\260727_0922_1b_funding_by_funding_year.csv

2020    5.053099e+07
2021    1.011812e+08
2022    1.501645e+08
2023    2.362088e+08
2024    3.544032e+08
2025    3.319381e+08
2026    2.328084e+08
2027    1.684541e+08
2028    9.821652e+07
2029    7.119121e+07
2030    1.829727e+07
2031    1.217275e+07
2032    5.454545e+06
2033    5.454545e+06
2034    0.000000e+00
2035    0.000000e+00
Name: Total funding (EUR), dtype: float64


### 1c. Total funding by region (2020-2025), plus an EU / EU+ / Europe comparison

The second table isn't a partition like the first - EU and EU+ are subsets of Europe, so their
totals are nested (EU <= EU+ <= Europe), not additive alongside the other regions.

In [42]:
from Funding_analysis_helpers import europe_tier_split

# Uses the Totals column directly (not the 2020-2035 per-year split) - many grants lack enough
# public info to know their duration, which would otherwise silently drop them from a "total".
region_split = split_multi_value(df, 'Funder region', delimiter=',')
region_split['weighted_funding'] = region_split[FUNDING_COL] * region_split['_weight']
by_region = region_split.groupby('Funder region_split')['weighted_funding'].sum().sort_values(ascending=False)
by_region.name = 'Total funding (2020-2025, EUR)'
save(by_region, '1c_funding_by_region')
print(by_region)

tier_split = europe_tier_split(df)
tier_split['weighted_funding'] = tier_split[FUNDING_COL] * tier_split['_weight']
by_tier = tier_split.groupby('_tiers')['weighted_funding'].sum().reindex(['EU', 'EU+', 'Europe'])
by_tier.name = 'Total funding (2020-2025, EUR)'
save(by_tier, '1c_funding_by_eu_tier')
print()
print(by_tier)

Saved 1c_funding_by_region -> data_output\analysis\260727_0922_1c_funding_by_region.csv
Funder region_split
Europe               1.297971e+09
North America        3.846055e+08
Asia/Asia Pacific    1.706996e+08
Middle East          2.745611e+07
Global               3.338193e+06
South America        2.421083e+06
Africa               5.599000e+05
Name: Total funding (2020-2025, EUR), dtype: float64
Saved 1c_funding_by_eu_tier -> data_output\analysis\260727_0922_1c_funding_by_eu_tier.csv

_tiers
EU        1.055625e+09
EU+       1.297821e+09
Europe    1.297971e+09
Name: Total funding (2020-2025, EUR), dtype: float64


### 2. Total funding by year x AP pillar, plus growth rates

In [43]:
# By START year (2020-2025), not the calculated per-year split stretching out to 2035 - each
# grant's full funding figure counts entirely toward the year it started, matching analysis 1a's
# methodology exactly but split by AP pillar.
by_start_year_pillar = df[df['Year project started'].between(2020, 2025)].pivot_table(
    index='AP pillar', columns='Year project started', values=FUNDING_COL, aggfunc='sum', fill_value=0
)
by_start_year_pillar.columns = by_start_year_pillar.columns.astype(int)
by_start_year_pillar = by_start_year_pillar[sorted(by_start_year_pillar.columns)]
by_start_year_pillar.loc['Total'] = by_start_year_pillar.sum()
save(by_start_year_pillar, '2a_funding_by_year_and_pillar')
print(by_start_year_pillar)

# YoY growth rate (matches the template's own formula: (year_N - year_N-1) / year_N-1)
growth_rates = by_start_year_pillar.apply(yoy_growth, axis=1)
growth_rates['Average growth rate (2021-2025)'] = growth_rates.iloc[:, 1:].mean(axis=1)
save(growth_rates, '2b_growth_rates_by_pillar')
print()
print(growth_rates)

Saved 2a_funding_by_year_and_pillar -> data_output\analysis\260727_0922_2a_funding_by_year_and_pillar.csv
Year project started          2020          2021          2022          2023  \
AP pillar                                                                      
Cross-cutting         1.082863e+07  3.758664e+07  3.817269e+07  9.888163e+07   
Cultivated            1.887046e+07  4.785970e+07  4.831153e+07  5.762773e+07   
Fermentation          2.175219e+07  4.266925e+07  4.716501e+07  1.166636e+08   
Plant-based           8.598882e+07  1.039077e+08  1.481198e+08  1.554902e+08   
Total                 1.374401e+08  2.320233e+08  2.817690e+08  4.286631e+08   

Year project started          2024          2025  
AP pillar                                         
Cross-cutting         1.456454e+08  2.754990e+07  
Cultivated            5.125342e+07  2.767294e+07  
Fermentation          1.652358e+08  1.499915e+08  
Plant-based           1.271253e+08  1.126404e+08  
Total                 4.892

### 2c/2d. Funding by year x region, and by EU / EU+ / Europe, with growth rates

In [44]:
# By START year (2020-2025), not the funding-year spread - matches 1a/2a's methodology,
# split by region. 1b is the one deliberate exception that stays funding-year-based.
by_year_region = region_split[region_split['Year project started'].between(2020, 2025)].pivot_table(
    index='Funder region_split', columns='Year project started', values='weighted_funding', aggfunc='sum', fill_value=0
)
by_year_region.columns = by_year_region.columns.astype(int)
by_year_region = by_year_region[sorted(by_year_region.columns)]
by_year_region.loc['Total'] = by_year_region.sum()
save(by_year_region, '2c_funding_by_year_and_region')
print(by_year_region)

growth_region = by_year_region.apply(yoy_growth, axis=1)
growth_region['Average growth rate (2021-2025)'] = growth_region.iloc[:, 1:].mean(axis=1)
save(growth_region, '2d_growth_rates_by_region')
print()
print(growth_region)

Saved 2c_funding_by_year_and_region -> data_output\analysis\260727_0922_2c_funding_by_year_and_region.csv
Year project started          2020          2021          2022          2023  \
Funder region_split                                                            
Africa                0.000000e+00  0.000000e+00  0.000000e+00  5.599000e+05   
Asia/Asia Pacific     1.244748e+07  5.770508e+07  1.684614e+07  2.121120e+07   
Europe                9.076646e+07  1.214811e+08  2.087149e+08  3.302847e+08   
Global                1.309406e+06  6.024936e+05  8.956450e+05  5.306482e+05   
Middle East           0.000000e+00  8.149300e+06  1.666206e+07  2.271742e+06   
North America         3.291247e+07  4.407532e+07  3.861254e+07  7.210791e+07   
South America         4.269000e+03  9.929000e+03  3.773600e+04  1.696999e+06   
Total                 1.374401e+08  2.320233e+08  2.817690e+08  4.286631e+08   

Year project started          2024          2025  
Funder region_split                       

In [45]:
by_year_tier = tier_split[tier_split['Year project started'].between(2020, 2025)].pivot_table(
    index='_tiers', columns='Year project started', values='weighted_funding', aggfunc='sum', fill_value=0
).reindex(['EU', 'EU+', 'Europe'])
by_year_tier.columns = by_year_tier.columns.astype(int)
by_year_tier = by_year_tier[sorted(by_year_tier.columns)]
save(by_year_tier, '2c_funding_by_year_and_eu_tier')
print(by_year_tier)

growth_tier = by_year_tier.apply(yoy_growth, axis=1)
growth_tier['Average growth rate (2021-2025)'] = growth_tier.iloc[:, 1:].mean(axis=1)
save(growth_tier, '2d_growth_rates_by_eu_tier')
print()
print(growth_tier)

Saved 2c_funding_by_year_and_eu_tier -> data_output\analysis\260727_0922_2c_funding_by_year_and_eu_tier.csv
Year project started          2020          2021          2022          2023  \
_tiers                                                                         
EU                    7.544736e+07  7.512691e+07  1.842527e+08  2.721124e+08   
EU+                   9.076646e+07  1.214811e+08  2.087149e+08  3.301348e+08   
Europe                9.076646e+07  1.214811e+08  2.087149e+08  3.302847e+08   

Year project started          2024          2025  
_tiers                                            
EU                    2.249562e+08  2.237289e+08  
EU+                   3.021287e+08  2.445947e+08  
Europe                3.021287e+08  2.445947e+08  
Saved 2d_growth_rates_by_eu_tier -> data_output\analysis\260727_0922_2d_growth_rates_by_eu_tier.csv

Year project started  2020      2021      2022      2023      2024      2025  \
_tiers                                                 

### 3. Total funding by country (2020-2025), with per-capita/GDP/R&D-spend

Fetches World Bank population/GDP and OECD R&D spend live, plus a live USD->EUR rate (Frankfurter)
to convert them - reused by analyses 4 and 8 below, so this cell's variables stay in scope.

In [46]:
country_split = split_multi_value(df, 'Funder Country', delimiter=',')
country_split['weighted_funding'] = country_split[FUNDING_COL] * country_split['_weight']

by_country = country_split.groupby('Funder Country_split')['weighted_funding'].sum().sort_values(ascending=False)
by_country.name = 'Total funding (2020-2025, EUR)'

print("Fetching World Bank population/GDP, OECD R&D spend, and a USD->EUR rate...")
POP_GDP_YEAR = 2023  # most recent year with broad World Bank/OECD coverage
pop = fetch_worldbank_indicator('SP.POP.TOTL', POP_GDP_YEAR)
gdp_usd = fetch_worldbank_indicator('NY.GDP.MKTP.CD', POP_GDP_YEAR)
gerd_usd = fetch_oecd_gerd(POP_GDP_YEAR)
usd_to_eur = fetch_usd_to_eur_rate(f'{POP_GDP_YEAR}-12-29')
print(f"USD->EUR rate ({POP_GDP_YEAR}-12-29): {usd_to_eur}")

by_country_df = by_country.reset_index().rename(columns={'Funder Country_split': 'Country'})
by_country_df['Population'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, pop))
by_country_df['GDP (EUR)'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, gdp_usd)) * usd_to_eur
by_country_df['R&D spend (EUR)'] = by_country_df['Country'].apply(
    lambda c: lookup_country_value(c, gerd_usd, COUNTRY_NAME_TO_OECD)
) * usd_to_eur
by_country_df['Funding per capita'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['Population']
by_country_df['Funding per GDP'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['GDP (EUR)']
by_country_df['GDP per capita'] = by_country_df['GDP (EUR)'] / by_country_df['Population']
by_country_df['Funding per R&D spend'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['R&D spend (EUR)']

n_no_population = by_country_df['Population'].isna().sum()
print(f"{n_no_population} entries have no population match (expected for non-country entities like 'EU'/'Global').")

save(by_country_df, '3_funding_by_country', index=False)
print(by_country_df.head(15).to_string())

Fetching World Bank population/GDP, OECD R&D spend, and a USD->EUR rate...


c:\Users\tsdav\Documents\Projects\SHEEP_Pipeline\Funding\Funding_analysis_helpers.py:146: DtypeWarning: Columns (0: OBS_STATUS, 1: Observation status, 2: OBS_STATUS_2, 3: Observation status 2, 4: OBS_STATUS_3, 5: Observation status 3, 6: AUX_OBS_STATUS, 7: Aux observation status, 8: CONF_STATUS, 9: Confidentiality status) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(io.StringIO(text))


USD->EUR rate (2023-12-29): 0.90498
2 entries have no population match (expected for non-country entities like 'EU'/'Global').
Saved 3_funding_by_country -> data_output\analysis\260727_0922_3_funding_by_country.csv
           Country  Total funding (2020-2025, EUR)   Population     GDP (EUR)  R&D spend (EUR)  Funding per capita  Funding per GDP  GDP per capita  Funding per R&D spend
0               EU                    4.527810e+08          NaN           NaN              NaN                 NaN              NaN             NaN                    NaN
1    United States                    3.217518e+08  336755052.0  2.516887e+13     8.616450e+11            0.955448         0.000013    74739.388482               0.000373
2          Denmark                    1.661546e+08    5946952.0  3.662017e+11     1.277912e+10           27.939458         0.000454    61578.048890               0.013002
3   United Kingdom                    1.545517e+08   68526000.0  3.095753e+12     1.037927e+11       

### 4. Total funding by country x AP pillar (2020-2025)

In [47]:
country_pillar = country_split.pivot_table(
    index='Funder Country_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
country_pillar['Total'] = country_pillar.sum(axis=1)
country_pillar = country_pillar.sort_values('Total', ascending=False)
save(country_pillar, '4_funding_by_country_and_pillar')
print(country_pillar)

Saved 4_funding_by_country_and_pillar -> data_output\analysis\260727_0922_4_funding_by_country_and_pillar.csv
AP pillar             Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder Country_split                                                            
EU                     3.877506e+07  3.108022e+07  1.853421e+08  1.975836e+08   
United States          9.587973e+07  4.123339e+07  1.341277e+08  5.051098e+07   
Denmark                6.085037e+06  2.890231e+06  6.449083e+07  9.268851e+07   
United Kingdom         4.169018e+07  2.754722e+07  4.095560e+07  4.435874e+07   
Netherlands            6.164645e+07  1.452487e+07  2.305151e+06  1.640810e+07   
Germany                4.181583e+06  9.593932e+06  7.620307e+06  6.360510e+07   
Singapore              1.285894e+07  1.798270e+07  3.960247e+07  1.269518e+07   
Finland                3.864689e+07  4.899000e+05  2.682890e+07  1.179841e+07   
Canada                 3.339732e+05  9.247841e+06  1.379242e+06  5.189263e+07   

### 4b. Total funding by region x AP pillar (2020-2025), and by EU / EU+ / Europe x AP pillar

In [48]:
region_pillar = region_split.pivot_table(
    index='Funder region_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
region_pillar['Total'] = region_pillar.sum(axis=1)
region_pillar = region_pillar.sort_values('Total', ascending=False)
save(region_pillar, '4b_funding_by_region_and_pillar')
print(region_pillar)

tier_pillar = tier_split.pivot_table(
    index='_tiers', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
).reindex(['EU', 'EU+', 'Europe'])
tier_pillar['Total'] = tier_pillar.sum(axis=1)
save(tier_pillar, '4b_funding_by_eu_tier_and_pillar')
print()
print(tier_pillar)

Saved 4b_funding_by_region_and_pillar -> data_output\analysis\260727_0922_4b_funding_by_region_and_pillar.csv
AP pillar            Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder region_split                                                            
Europe                2.257344e+08  1.119103e+08  3.607699e+08  5.995560e+08   
North America         9.621371e+07  5.048123e+07  1.355070e+08  1.024036e+08   
Asia/Asia Pacific     2.677911e+07  6.841370e+07  4.552182e+07  2.998501e+07   
Middle East           9.491092e+06  1.750923e+07  4.557900e+05  0.000000e+00   
Global                0.000000e+00  1.880443e+06  6.630000e+05  7.947500e+05   
South America         4.465736e+05  1.441711e+06  0.000000e+00  5.327980e+05   
Africa                0.000000e+00  0.000000e+00  5.599000e+05  0.000000e+00   

AP pillar                   Total  
Funder region_split                
Europe               1.297971e+09  
North America        3.846055e+08  
Asia/Asia Pacific    1.70

### 5. Top 20 funder institutions, with yearly split (no pillar split)

In [49]:
# Yearly breakdown by START year (2020-2025), not the funding-year spread - same default as
# everywhere else except 1b. Total stays the funder's full Totals-column sum (not window-scoped),
# same as every other "Total" in this notebook.
by_funder_year = df[df['Year project started'].between(2020, 2025)].pivot_table(
    index='Funder name', columns='Year project started', values=FUNDING_COL, aggfunc='sum', fill_value=0
)
by_funder_year.columns = by_funder_year.columns.astype(int)
by_funder_year = by_funder_year[sorted(by_funder_year.columns)]
by_funder_year['Total'] = df.groupby('Funder name')[FUNDING_COL].sum()
by_funder_year = by_funder_year.fillna(0)
top20_funders = by_funder_year.sort_values('Total', ascending=False).head(20)
save(top20_funders, '5_top20_funders')
print(top20_funders)

Saved 5_top20_funders -> data_output\analysis\260727_0922_5_top20_funders.csv
Year project started                                        2020  \
Funder name                                                        
European Commission                                 3.741279e+07   
UK Research and Innovation (UKRI)                   3.111053e+06   
Novo Nordisk Foundation                             3.345500e+04   
Bezos Earth Fund (BEF)                              0.000000e+00   
Business Finland                                    3.272000e+06   
Dutch National Growth Fund                          0.000000e+00   
Department of Commerce (DoC)                        0.000000e+00   
Research Council of Norway                          1.022493e+07   
"Agency for Science, Technology, and Research (...  0.000000e+00   
Federal Ministry of Food and Agriculture (BMEL)     1.552889e+06   
Protein Industries Canada                           1.955580e+07   
Defense Advanced Research Projects Age

### 6. Top 20 country rankings per year, 2020-2025 (non-cumulative)

Two shapes: (a) rank 1-20 x year, country name per cell (for a table); (b) country x year, rank
number per cell (for graphing a ranked-line/bump chart). Distinct from analysis #8's cumulative
ranking below - confirmed via the template these are two different things, not variants of one.

In [50]:
# By START year (2020-2025), not the funding-year spread - same default as everywhere else in
# this notebook except 1b.
annual_country = country_split[country_split['Year project started'].between(2020, 2025)]
year_country_totals = {}
for y in range(2020, 2026):
    year_country_totals[y] = annual_country[annual_country['Year project started'] == y].groupby(
        'Funder Country_split'
    )['weighted_funding'].sum().sort_values(ascending=False)

# Shape A: rank (1-20) x year, country name per cell
rank_table = pd.DataFrame({y: year_country_totals[y].head(20).index.tolist() for y in range(2020, 2026)})
rank_table.index = pd.RangeIndex(1, 21, name='Rank')
save(rank_table, '6a_country_rank_table')
print(rank_table)

# Shape B: country x year, rank number per cell (for graphing)
all_countries_ranked = sorted(set().union(*[year_country_totals[y].head(20).index for y in range(2020, 2026)]))
country_rank_grid = pd.DataFrame(index=all_countries_ranked, columns=range(2020, 2026), dtype='float')
for y in range(2020, 2026):
    ranks = {c: i + 1 for i, c in enumerate(year_country_totals[y].index)}
    for c in all_countries_ranked:
        country_rank_grid.loc[c, y] = ranks.get(c, np.nan)
save(country_rank_grid, '6b_country_rank_grid')
print()
print(country_rank_grid)

Saved 6a_country_rank_table -> data_output\analysis\260727_0922_6a_country_rank_table.csv
                2020            2021            2022            2023  \
Rank                                                                   
1                 EU       Singapore              EU              EU   
2             Canada          Norway         Denmark     Netherlands   
3             Norway   United States   United States   United States   
4            Germany              EU  United Kingdom  United Kingdom   
5             Sweden         Denmark          Israel         Germany   
6      United States           Spain     South Korea         Denmark   
7        New Zealand          Canada         Germany          France   
8            Denmark          Israel           Spain          Canada   
9            Belgium  United Kingdom         Finland           Japan   
10           Finland          Sweden     Switzerland           Spain   
11    United Kingdom       Australia          

### 7. Total funding 2020-2025 by AP pillar x End product type, plus a separate Dairy sub-split

In [51]:
ep_split = split_multi_value(df, 'End product type', delimiter=';')
ep_split['weighted_funding'] = ep_split[FUNDING_COL] * ep_split['_weight']
pillar_end_product = ep_split.pivot_table(
    index='End product type_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
pillar_end_product['Total'] = pillar_end_product.sum(axis=1)
save(pillar_end_product, '7a_funding_by_pillar_and_end_product')
print(pillar_end_product)

# Dairy's own constituents, not cross-tabulated with pillar. Starts from ep_split's Dairy share
# (not df directly) so a 'Meat; Dairy' grant contributes the same weighted share here as it does
# to 7a's Dairy total, then splits that share further across sub-end products. 'sub-end product'
# is exclusively populated for Dairy rows (confirmed: no non-Dairy row has one) - Dairy rows
# missing a sub-end product are labelled 'Agnostic' here, for this breakdown only, so their
# funding isn't silently dropped. Not written back to the underlying data: 'Agnostic' is already a
# distinct End product type value, and reusing it as a sub-end product value there would be confusing.
dairy_ep = ep_split[ep_split['End product type_split'] == 'Dairy'].copy()
dairy_ep['sub-end product'] = dairy_ep['sub-end product'].fillna('Agnostic')
dairy_ep = dairy_ep.rename(columns={'weighted_funding': 'dairy_weighted_funding'})
sub_ep_split = split_multi_value(dairy_ep, 'sub-end product', delimiter=';')
sub_ep_split['weighted_funding'] = sub_ep_split['dairy_weighted_funding'] * sub_ep_split['_weight']
dairy_breakdown = sub_ep_split.groupby('sub-end product_split')['weighted_funding'].sum().sort_values(ascending=False)
dairy_breakdown.name = 'Total funding (2020-2025, EUR)'
save(dairy_breakdown, '7b_dairy_sub_breakdown')
print()
print(dairy_breakdown)

Saved 7a_funding_by_pillar_and_end_product -> data_output\analysis\260727_0922_7a_funding_by_pillar_and_end_product.csv
AP pillar                               Cross-cutting    Cultivated  \
End product type_split                                                
Agnostic                                 3.178000e+08  5.915520e+07   
Chocolate, desserts, and confectionery   0.000000e+00  3.597543e+05   
Dairy                                    5.628772e+06  1.185138e+06   
Eggs and egg proteins                    1.484131e+06  0.000000e+00   
Fish and seafood                         1.600180e+05  2.656016e+07   
Meat                                     3.359196e+07  1.643764e+08   
Spreads, sauces, and condiments          0.000000e+00  0.000000e+00   

AP pillar                               Fermentation   Plant-based  \
End product type_split                                               
Agnostic                                4.613802e+08  5.259410e+08   
Chocolate, desserts, and confe

### 8. Investment and cumulative investment by country, 2020-2025, with cumulative rank

Reuses `year_country_totals` computed for analysis #6 above.

In [52]:
investment_by_year = pd.DataFrame({y: year_country_totals[y] for y in range(2020, 2026)}).fillna(0)
save(investment_by_year, '8a_investment_by_country_and_year')

cumulative_by_year = investment_by_year.cumsum(axis=1)
save(cumulative_by_year, '8b_cumulative_investment_by_country_and_year')

# Rank 1 = highest cumulative total-to-date that year
cumulative_rank = cumulative_by_year.rank(axis=0, ascending=False, method='min')
save(cumulative_rank, '8c_cumulative_investment_rank')

print(investment_by_year.head(10))
print()
print(cumulative_by_year.head(10))
print()
print(cumulative_rank.head(10))

Saved 8a_investment_by_country_and_year -> data_output\analysis\260727_0922_8a_investment_by_country_and_year.csv
Saved 8b_cumulative_investment_by_country_and_year -> data_output\analysis\260727_0922_8b_cumulative_investment_by_country_and_year.csv
Saved 8c_cumulative_investment_rank -> data_output\analysis\260727_0922_8c_cumulative_investment_rank.csv
                              2020          2021          2022          2023  \
Funder Country_split                                                           
Australia             2.060400e+04  4.828994e+06  1.328364e+06  6.063202e+05   
Austria               0.000000e+00  0.000000e+00  6.654340e+05  3.200000e+06   
Belgium               3.600000e+06  0.000000e+00  0.000000e+00  5.995271e+06   
Brazil                0.000000e+00  0.000000e+00  0.000000e+00  1.473885e+06   
Canada                2.346367e+07  9.627245e+06  5.676812e+06  1.334143e+07   
Chile                 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Chin

### 9. Research category as a % of total funding, by region (+ EU / EU+ / Europe) and by country - overall, and by AP pillar

`'Research category'` is multi-value like `'Funder Country'`/`'Funder region'`, so it gets the
same fractional split - a grant tagged with 3 categories has each get 1/3 of its (region- or
country-weighted) share. Grants with no `'Research category'` at all are excluded entirely, so
`'Total (EUR)'` here is the classified-funding total, not the full region/country total from
1c/3 - percentages are a share of classified funding only.

In [53]:
def category_pct_table(base_split, index_col, weighted_col_name):
    # Grants with no Research category are dropped entirely (not labelled 'Unclassified') -
    # percentages here are a share of classified funding only, so Total (EUR) will be less than
    # the region/country's true total wherever some grants lack a category.
    base = base_split.rename(columns={'weighted_funding': weighted_col_name}).copy()
    exploded = split_multi_value(base, 'Research category', delimiter=',')
    exploded['weighted_funding'] = exploded[weighted_col_name] * exploded['_weight']
    eur = exploded.pivot_table(
        index=index_col, columns='Research category_split', values='weighted_funding', aggfunc='sum', fill_value=0
    )
    eur['Total (EUR)'] = eur.sum(axis=1)
    pct = eur.drop(columns='Total (EUR)').div(eur['Total (EUR)'], axis=0) * 100
    pct['Total (EUR)'] = eur['Total (EUR)']
    return pct.sort_values('Total (EUR)', ascending=False)


def category_pct_by_pillar(base_split, index_col, weighted_col_name):
    pillars = sorted(base_split['AP pillar'].dropna().unique())
    return pd.concat(
        {p: category_pct_table(base_split[base_split['AP pillar'] == p], index_col, weighted_col_name) for p in pillars},
        names=['AP pillar']
    )

In [54]:
region_category_pct = category_pct_table(region_split, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct, '9a_research_category_pct_by_region')
print(region_category_pct)

region_category_pct_by_pillar = category_pct_by_pillar(region_split, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct_by_pillar, '9a_research_category_pct_by_region_and_pillar')
print()
print(region_category_pct_by_pillar)

Saved 9a_research_category_pct_by_region -> data_output\analysis\260727_0922_9a_research_category_pct_by_region.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder region_split                                              
Europe                           10.455913            2.348816   
North America                     4.902356            2.338578   
Asia/Asia Pacific                21.548814            7.814260   
Middle East                       0.000000            0.000000   
Global                           26.362429           12.716364   
South America                     0.000000            0.000000   
Africa                           20.000000            0.000000   

Research category_split  Cell line development  Consumer & market research  \
Funder region_split                                                          
Europe                                1.319316                    5.563653   
North America                         1.594586         

In [55]:
tier_category_pct = category_pct_table(tier_split, '_tiers', 'tier_weighted_funding').reindex(['EU', 'EU+', 'Europe'])
save(tier_category_pct, '9a_research_category_pct_by_eu_tier')
print(tier_category_pct)

tier_category_pct_by_pillar = category_pct_by_pillar(tier_split, '_tiers', 'tier_weighted_funding')
save(tier_category_pct_by_pillar, '9a_research_category_pct_by_eu_tier_and_pillar')
print()
print(tier_category_pct_by_pillar)

Saved 9a_research_category_pct_by_eu_tier -> data_output\analysis\260727_0922_9a_research_category_pct_by_eu_tier.csv
Research category_split  Bioprocess design  Cell culture media  \
_tiers                                                           
EU                               10.218250            1.552240   
EU+                              10.457121            2.337530   
Europe                           10.455913            2.348816   

Research category_split  Cell line development  Consumer & market research  \
_tiers                                                                       
EU                                    0.908419                    5.457232   
EU+                                   1.319468                    5.564296   
Europe                                1.319316                    5.563653   

Research category_split  Crop development  End product formulation  \
_tiers                                                               
EU                  

In [56]:
country_category_pct = category_pct_table(country_split, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct, '9b_research_category_pct_by_country')
print(country_category_pct)

country_category_pct_by_pillar = category_pct_by_pillar(country_split, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct_by_pillar, '9b_research_category_pct_by_country_and_pillar')
print()
print(country_category_pct_by_pillar)

Saved 9b_research_category_pct_by_country -> data_output\analysis\260727_0922_9b_research_category_pct_by_country.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder Country_split                                             
EU                               12.863224            1.738614   
United States                     5.344932            2.227554   
Denmark                          12.516476            0.516037   
United Kingdom                   12.404175            8.325029   
Netherlands                       4.777105            2.669911   
Germany                           4.881928            0.685060   
Singapore                        15.089249            3.764120   
Finland                          14.454670            0.314991   
Canada                            2.636785            2.906920   
Norway                           10.121437            1.736356   
Sweden                            1.863944            0.222507   
Spain                   